# CareerLens AI

## Notebook 02 — Feature Engineering

### Objective

The purpose of this notebook is to prepare resume text for machine learning.

In this notebook we will:

- Load the dataset
- Clean resume text
- Encode job categories
- Convert resume text into numerical features using TF-IDF
- Split the dataset into training and testing sets
- Save processed artifacts for model training

---

**Author:** Anushka Das  
**Project:** CareerLens AI  
**Notebook:** 02_feature_engineering.ipynb  
**Version:** 1.0

In [1]:
# ==========================================================
# Import Required Libraries
# ==========================================================

import re
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries imported successfully.")

Libraries imported successfully.


# Load Dataset

In [2]:
# ==========================================================
# Load Dataset
# ==========================================================

dataset_path = "../datasets/raw/Resume.csv"

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset loaded successfully.
Rows: 2484
Columns: 4


In [3]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


# Text Preprocessing

We will clean the resume text before converting it into machine learning features.

For Version 1, the cleaning process includes:

- Lowercasing text
- Removing URLs
- Removing email addresses
- Removing phone-number-like patterns
- Removing non-alphabetic characters
- Removing extra spaces

In [4]:
# ==========================================================
# Text Cleaning Function
# ==========================================================

def clean_resume(text):
    """
    Clean resume text for NLP feature engineering.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\+?\d[\d\s()-]{8,}\d", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [5]:
# Apply cleaning

df["cleaned_resume"] = df["Resume_str"].apply(clean_resume)

df[["Resume_str", "cleaned_resume"]].head()

,Resume_str,cleaned_resume
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,hr administrator marketing associate hr admini...
1,"HR SPECIALIST, US HR OPERATIONS ...",hr specialist us hr operations summary versati...
2,HR DIRECTOR Summary Over 2...,hr director summary over years experience in r...
3,HR SPECIALIST Summary Dedica...,hr specialist summary dedicated driven and dyn...
4,HR MANAGER Skill Highlights ...,hr manager skill highlights hr skills hr depar...


# Target Label Encoding

Machine learning models cannot directly understand text labels such as `HR`, `AVIATION`, or `INFORMATION-TECHNOLOGY`.

We will convert each category into a numerical label.

In [6]:
# ==========================================================
# Encode Target Labels
# ==========================================================

label_encoder = LabelEncoder()

df["target"] = label_encoder.fit_transform(df["Category"])

print("Labels encoded successfully.")
print(f"Number of classes: {len(label_encoder.classes_)}")

Labels encoded successfully.
Number of classes: 24


In [7]:
# View category to encoded label mapping

label_mapping = pd.DataFrame({
    "Category": label_encoder.classes_,
    "Encoded_Label": range(len(label_encoder.classes_))
})

label_mapping

,Category,Encoded_Label
0,ACCOUNTANT,0
1,ADVOCATE,1
2,AGRICULTURE,2
3,APPAREL,3
4,ARTS,4
5,AUTOMOBILE,5
6,AVIATION,6
7,BANKING,7
8,BPO,8
9,BUSINESS-DEVELOPMENT,9


# TF-IDF Feature Engineering

TF-IDF converts text into numerical features.

It gives importance to words that appear frequently in a document but are not too common across all documents.

For Version 1, TF-IDF is a strong baseline because it is simple, interpretable, and works well for text classification.

In [8]:
# ==========================================================
# TF-IDF Vectorization
# ==========================================================

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2)
)

X = tfidf_vectorizer.fit_transform(df["cleaned_resume"])
y = df["target"]

print("TF-IDF vectorization completed.")
print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

TF-IDF vectorization completed.
Feature matrix shape: (2484, 5000)
Target shape: (2484,)


# Train-Test Split

We split the dataset into training and testing sets.

The training set will be used to train models.

The testing set will be used to evaluate how well the model performs on unseen resumes.

In [9]:
# ==========================================================
# Train-Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train-test split completed.")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape : {y_test.shape}")

Train-test split completed.
X_train shape: (1987, 5000)
X_test shape : (497, 5000)
y_train shape: (1987,)
y_test shape : (497,)


# Save Processed Artifacts

We will save:

- Cleaned dataset
- TF-IDF vectorizer
- Label encoder
- Train/test split files

These files will be reused during model training.

In [10]:
# ==========================================================
# Save Processed Dataset and Artifacts
# ==========================================================

processed_dataset_path = "../datasets/processed/processed_resumes.csv"
vectorizer_path = "../models/tfidf_vectorizer.pkl"
label_encoder_path = "../models/label_encoder.pkl"

df.to_csv(processed_dataset_path, index=False)

joblib.dump(tfidf_vectorizer, vectorizer_path)
joblib.dump(label_encoder, label_encoder_path)

print("Processed dataset saved.")
print("TF-IDF vectorizer saved.")
print("Label encoder saved.")

Processed dataset saved.
TF-IDF vectorizer saved.
Label encoder saved.


In [11]:
# Save train/test split

joblib.dump(X_train, "../datasets/processed/X_train.pkl")
joblib.dump(X_test, "../datasets/processed/X_test.pkl")
joblib.dump(y_train, "../datasets/processed/y_train.pkl")
joblib.dump(y_test, "../datasets/processed/y_test.pkl")

print("Train/test split saved successfully.")

Train/test split saved successfully.


# Day 4 Final Observations

## What we completed

- Loaded the resume dataset.
- Created a reusable text cleaning function.
- Cleaned resume text.
- Encoded resume categories into numerical labels.
- Created TF-IDF features.
- Split data into training and testing sets.
- Saved processed data and preprocessing artifacts.

## Engineering Decision

For Version 1, CareerLens AI will use:

- Regex-based text preprocessing
- TF-IDF vectorization
- Encoded job categories
- Stratified train-test split

## Next Step

The next notebook will focus on baseline machine learning model training.